# Hindi Restoration + Translation - executed demo

This notebook is the **already-executed evidence notebook** for the project. It
shows real predictions, real output files, and the measured scores from a local
run with trained LoRA adapters and a local IndicTrans2 copy.

> **Colab reviewers: this notebook is read-only.** The model weights and
> processed data are intentionally not committed to the repository. Read the
> saved outputs below. If you click **Run all** in Colab, model-loading cells
> will fail because those local artifacts are absent. To run the project from
> scratch, use `notebooks/colab_end_to_end.ipynb`, which clones the repo and
> trains the models on a T4 GPU.

**It does not train anything.** For the full from-scratch run (data preparation,
training, evaluation and the pipeline), open
`notebooks/colab_end_to_end.ipynb`.

## What the project does

The input is Hindi text that has lost its punctuation, and its sentences are in
the wrong order. The project:

1. restores punctuation and sentence boundaries,
2. recovers a sensible sentence order,
3. translates the result into English,
4. keeps every five sentences together as a group.

## The three stages

| Stage | Question | Model | What you will see here |
| --- | --- | --- | --- |
| 1. Punctuation | Where do commas and full stops go? | mBERT + LoRA | Predicted labels compared with gold labels |
| 2. Ordering | Which sentence comes first? | mBERT + LoRA | A before/after probability for one pair |
| 3. Translation | How do we say it in English? | IndicTrans2 | Three Hindi sentences translated |
| Pipeline | How do the stages fit together? | all three | Seven samples with aligned Hindi/English files |

## How to use this notebook

- Read the saved outputs; they are the evidence artifact.
- **Do not use Run all in Colab** unless you have the local model and data
  artifacts. They are not part of the public repository.
- To reproduce the run, clone the repo, prepare the data, train the adapters,
  and provide a local IndicTrans2 copy as described in the README.
- If a model cannot be loaded, the cell fails loudly. There is no hidden fallback model.

**Environment used for the saved outputs:** Python 3.12, torch 2.15 on Apple
MPS, transformers 4.44.2, peft 0.13.2. The full run takes roughly 2-3 minutes.

## Important words

| Word | Meaning |
| --- | --- |
| **Model** | A program with learned weights that makes predictions. |
| **Inference** | Using a trained model on new text (no training). |
| **Adapter** | The small LoRA weights saved on top of mBERT. |
| **Label** | The punctuation answer for a word. |
| **Token** | A piece of text the model reads; a word may have several. |
| **Kendall's tau** | A score for how similar two sentence orders are. |
| **BLEU / chrF** | Standard translation quality scores. |


## 0. Set the project folder

This cell is only needed for a local reproduction run. It finds the project root
from either the repository root or the `notebooks/` folder, changes into it, and
adds it to Python's import path.

**Colab note:** finding the folder does not provide the missing model weights or
processed data. The executed outputs below are the intended review path.


In [1]:
from pathlib import Path
import os, sys, json

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)
print('Python:', sys.version.split()[0])


Project root: /Users/vaibhavparmar/Documents/nlp-zero-to-hero
Python: 3.12.13


### What you should see

`Project root` should end with `nlp-zero-to-hero`, and the Python version should
match the environment you are using.

If the import path is wrong, every later `from src...` import fails. This cell is
the fix for that.

## 1. Punctuation restoration

### The problem

The raw input is a bag of Hindi words with no punctuation:

```text
भारत एक विशाल देश है यह एक परीक्षण है
```

A human can see two sentences, but the model must learn where the first one
ends.

### The labels

The model predicts one label per word:

| Label | Mark | Meaning |
| --- | --- | --- |
| `none` | nothing | inside a sentence |
| `comma` | `,` | comma follows |
| `sentence_end` | `।` | sentence ends here |
| `question` | `?` | question ends here |
| `exclamation` | `!` | exclamation ends here |

A Hindi word can become several **subword tokens** in mBERT. The label is placed
on the first subword, and the remaining subwords are ignored during training.

### What is LoRA?

LoRA freezes the large pretrained mBERT and trains a tiny set of extra weights.
Those weights are the **adapter** loaded here. This is why a 178M-parameter
model can run inside a small memory budget.

### What this cell does

1. Loads one example from the held-out punctuation test set.
2. Loads the punctuation adapter.
3. Predicts a label for every word.
4. Compares predictions with the gold labels.
5. Prints the first 12 words and the window accuracy.

In [2]:
from src.constants import ID2LABEL, LABEL2ID
from src.punctuation.predict import PunctuationRestorer
from src.utils import read_jsonl

row = read_jsonl('data/processed/punctuation/test.jsonl')[0]
restorer = PunctuationRestorer('models/punctuation')
predicted = restorer.predict_labels(row['words'])
truth = [LABEL2ID[label] for label in row['labels']]
accuracy = sum(p == t for p, t in zip(predicted, truth)) / len(truth)
print(f'words in window: {len(truth)}')
for word, true_id, pred_id in list(zip(row['words'], truth, predicted))[:12]:
    print(f'{word:18s} true={ID2LABEL[true_id]:12s} pred={ID2LABEL[pred_id]}')
print(f'window token accuracy: {accuracy:.3f}')
restorer.close()


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


words in window: 73
अपने               true=none         pred=none
तरह                true=none         pred=none
का                 true=none         pred=none
राज्य-प्रबन्धन     true=none         pred=none
विषयक              true=none         pred=none
यह                 true=none         pred=none
प्राचीनतम          true=none         pred=none
ग्रन्थ             true=none         pred=none
है                 true=sentence_end pred=sentence_end
इसमें              true=none         pred=none
राज्यव्यवस्था      true=comma        pred=comma
कृषि               true=comma        pred=comma
window token accuracy: 0.849


### How to read the output

Each line has three parts:

```text
word               true=<gold label>   pred=<model label>
```

The goal is for `true` and `pred` to match. The printed window accuracy is the
fraction of words in this one window that were correct; it is not the official
test score. The official score is computed over many windows in the evaluation
section.

The warning about "newly initialized" classifier weights is expected: the
classification head is part of the saved adapter, and Hugging Face prints this
message while loading the base model.

## 2. Sentence ordering

### The problem

After punctuation we have sentences, but they are shuffled. A 20-sentence
article has more than 2 quintillion possible orders, so predicting the whole
permutation directly is impractical.

### The pairwise idea

Instead, the model answers a smaller question:

> Given sentence A and sentence B, does A come before B?

It is trained on consecutive Wikipedia sentences:

- `(A, B) -> 1` when A really comes first,
- `(B, A) -> 0` for the swapped pair.

At inference, the model produces a probability for many pairs. A **decoder**
turns those pairwise opinions into one order. The default decoder is **Borda**:
give each sentence an average "before the others" score, then sort.

### What this cell does

1. Loads one held-out ordering pair.
2. Loads the ordering adapter.
3. Builds the 2x2 score matrix.
4. Prints `P(A before B)` and `P(B before A)`.

The two probabilities should add to 1.0, and the gold label tells us which
direction is correct.

In [3]:
from src.ordering.decode import OrderScorer
from src.utils import read_jsonl

pair = read_jsonl('data/processed/ordering/test.jsonl')[0]
scorer = OrderScorer('models/ordering')
matrix = scorer.score_matrix([pair['sentence_a'], pair['sentence_b']])
print('gold label:', pair['label'], '(1 = A before B)')
print('P(A before B):', round(float(matrix[0, 1]), 3))
print('P(B before A):', round(float(matrix[1, 0]), 3))
print('A:', pair['sentence_a'][:120])
print('B:', pair['sentence_b'][:120])
scorer.close()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


gold label: 1 (1 = A before B)
P(A before B): 0.711
P(B before A): 0.289
A: यह बाद में ऑगस्ट कॉम्ट(1798-1857 द्वारा 1838 में स्थापित किया गया
B: इससे पहले कॉम्ट ने सामाजिक भौतिकी शब्द का इस्तेमाल किया था लेकिन बाद में वह दूसरों द्वारा अपनाया गया विशेष रूप से बेल्जि


### How to read the output

- `gold label: 1` means sentence A really comes before sentence B.
- `P(A before B)` above `0.5` means the model agrees with the gold label.
- `P(B before A)` is the complement, so it should be below `0.5`.

A single pair is only a sanity check. The real ordering quality is measured with
**Kendall's tau** on complete articles in the evaluation section.

## 3. Hindi to English translation

**IndicTrans2** is a sequence-to-sequence model: it reads a Hindi sentence and
generates an English sentence token by token. The 200M version is small enough
to run locally.

### Why the toolkit?

IndicTrans2 expects language tags such as `hin_Deva` and `eng_Latn`, plus
language-specific preprocessing. `IndicTransToolkit` adds those tags and cleans
up the generated text. The wrapper in `src/translation/indictrans.py` handles
this for us.

There is **no fallback translator**. If the model cannot be loaded, the cell
raises an error instead of quietly switching to a different model.

### What this cell does

It translates three short Hindi sentences and prints the Hindi/English pair for
each one. The exact wording can vary slightly with beams and model versions, so
do not expect one fixed string.

In [4]:
from src.translation.indictrans import IndicTrans2Translator

translator = IndicTrans2Translator('models/translation/indictrans2-indic-en-200M')
sentences = [
    'भारत एक विशाल देश है।',
    'यह एक परीक्षण है।',
    'कृपया अपनी रिपोर्ट कल तक जमा करें।',
]
for hi, en in zip(sentences, translator.translate(sentences)):
    print(f'HI: {hi}')
    print(f'EN: {en}\n')
translator.close()


HI: भारत एक विशाल देश है।
EN: India is a huge country.

HI: यह एक परीक्षण है।
EN: It's a test.

HI: कृपया अपनी रिपोर्ट कल तक जमा करें।
EN: Please submit your report by tomorrow.



### What you should see

One non-empty English line for each Hindi line. The saved outputs are:

- `भारत एक विशाल देश है।` -> `India is a huge country.`
- `यह एक परीक्षण है।` -> `It's a test.`
- `कृपया अपनी रिपोर्ट कल तक जमा करें।` -> `Please submit your report by tomorrow.`

Your wording may differ slightly; that is normal for a generative model.

## 4. End-to-end output files

Training and the full pipeline have already run, so `outputs/` and `reports/`
exist. This section checks the most important property: **alignment**.

For every sample the pipeline writes four files:

| File | Contents |
| --- | --- |
| `sample-N.sentences_hi.txt` | Hindi sentences, one per line, in recovered order |
| `sample-N.groups_hi.txt` | The same sentences grouped five at a time |
| `sample-N.sentences_en.txt` | English translations, one per line |
| `sample-N.groups_en.txt` | The same translations grouped five at a time |

Sentence `i` in the Hindi file must match sentence `i` in the English file. If
the counts drift, the groups are meaningless.

In [5]:
summary = json.loads(Path('reports/run_summary.json').read_text())
print('timings (seconds):', summary['timings_seconds'])
for stem, info in summary['files'].items():
    hi = Path(f'outputs/{stem}.sentences_hi.txt').read_text().splitlines()
    en = Path(f'outputs/{stem}.sentences_en.txt').read_text().splitlines()
    gh = Path(f'outputs/{stem}.groups_hi.txt').read_text().splitlines()
    ge = Path(f'outputs/{stem}.groups_en.txt').read_text().splitlines()
    ok = len(hi) == len(en) == info['sentences'] and len(gh) == len(ge) == info['groups']
    print(f"{stem}: {info['sentences']} sentences, {info['groups']} groups, aligned={ok}")
print('\nFirst 3 ordered Hindi sentences:')
for line in Path('outputs/sample-1.sentences_hi.txt').read_text().splitlines()[:3]:
    print('-', line)
print('\nFirst 3 English translations:')
for line in Path('outputs/sample-1.sentences_en.txt').read_text().splitlines()[:3]:
    print('-', line)


timings (seconds): {'punctuation': 2.31, 'ordering': 130.53, 'translation': 102.17}
sample-1: 52 sentences, 11 groups, aligned=True
sample-2: 55 sentences, 11 groups, aligned=True
sample-3: 49 sentences, 10 groups, aligned=True
sample-4: 53 sentences, 11 groups, aligned=True
sample-5: 55 sentences, 11 groups, aligned=True
sample-6: 80 sentences, 16 groups, aligned=True
sample-7: 99 sentences, 20 groups, aligned=True

First 3 ordered Hindi sentences:
- तब, महालक्ष्मी हाथ में कमल धारण किए हुए, उनके सामने प्रकट हुई थीं।
- इसका नाम बनारस या वाराणसी भी है।
- वे इस सोच में पड़ गए, कि युद्ध में मारे गए, असंख्य, कुटुम्बियों, बंधुओं, व निर्दोष, लोगों की आत्मा की शांति के लिए क्या उपाय या संस्कार किया जाए।

First 3 English translations:
- Then, Mahalakshmi appeared before him, holding a lotus in her hand.
- It is also called Banaras or Varanasi.
- They wondered what measures or rites should be taken for the peace of the souls of the countless, relatives, brothers, and innocent people who died in

### How to read the output

For each sample, `aligned=True` means:

- the reported number of sentences equals the number of lines in the Hindi and
  English sentence files,
- the number of groups equals the number of lines in both group files.

The printed Hindi and English sentences let you eyeball the result. A good sign
is that the Hindi has punctuation and the English reads like natural text; the
ordering stage is the weakest, so some sentences may still be out of place.

## 5. Evaluation: the honest scores

The numbers in `reports/evaluation.json` come from held-out data, not from the
examples the models trained on.

| Metric | What it means |
| --- | --- |
| **Punctuation accuracy** | Fraction of words with the correct label. High because `none` is common. |
| **Boundary F1** | How well the model finds commas, sentence ends, question and exclamation marks. This is the meaningful punctuation score. |
| **Ordering pair accuracy** | How often the model puts a pair in the right order. `0.5` is guessing. |
| **Kendall's tau** | How similar the full recovered order is to the true order. |
| **BLEU / chrF** | Translation scores. They are comparative, not percentages. |

These adapters were trained on small local subsets because Apple MPS is slower
than a Colab T4. The full Colab recipe uses all 25,000 punctuation windows and
40,000 ordering pairs, so its ordering score should be better.

In [6]:
ev = json.loads(Path('reports/evaluation.json').read_text())
print('punctuation accuracy / boundary F1:', ev['punctuation']['accuracy'], '/', ev['punctuation']['boundary']['f1'])
print('ordering pair accuracy:', ev['ordering']['pairs']['accuracy'])
print('ordering Borda Kendall tau:', ev['ordering']['kendall_tau']['borda']['mean_tau'])
print('translation BLEU / chrF:', ev['translation']['bleu'], '/', ev['translation']['chrf'])


punctuation accuracy / boundary F1: 0.8724 / 0.8842
ordering pair accuracy: 0.6543
ordering Borda Kendall tau: 0.2007
translation BLEU / chrF: 34.125 / 59.686


### How to read the scores

- Punctuation accuracy `0.8724` and boundary F1 `0.8842` mean the punctuation
  stage is working well on the test windows.
- Ordering pair accuracy `0.6543` is clearly above `0.5`, but Kendall's tau
  `0.2007` shows the full article order is only partly recovered. This is the
  hardest stage and the honest weak point of the local run.
- Translation BLEU `34.125` and chrF `59.686` are reasonable for a 200M model
  on 100 IITB sentences.

Do not compare BLEU across different datasets; compare it across model versions
on the same test set.

## 6. Reproduce and verify from the command line

The notebook is a demonstration; the project is driven by command-line modules.
These commands reproduce the same artifacts and then run the full self-check.

The first checks are cheap: they load one example and confirm that the adapter
files exist and produce valid predictions. The final check verifies every output
file. If any check fails, it prints the exact missing file or bad metric.

```bash
python -m src.pipeline \
  --translation-model models/translation/indictrans2-indic-en-200M \
  --translation-batch-size 8

python -m src.evaluate \
  --translation-model models/translation/indictrans2-indic-en-200M \
  --max-translation-examples 100

python scripts/verify_colab_run.py --allow-skipped-translation
```

## 7. What to read next

This notebook shows the finished pipeline. To understand how it was built, read
the code in this order:

1. `src/constants.py` - paths, labels, model names and settings.
2. `src/pipeline.py` - the three stages connected end to end.
3. `src/punctuation/` - how words become labels and labels become sentences.
4. `src/ordering/` - pairs, pair training and the Borda/clusters/beam decoders.
5. `src/translation/indictrans.py` - the IndicTrans2 wrapper.
6. `src/data_prep.py` and `src/evaluate.py` - how data and metrics are built.

Then open `notebooks/colab_end_to_end.ipynb`. It starts from zero: environment,
data preparation, training, evaluation and the full pipeline, with an
explanation above every step.

### Exercises

- Open `outputs/sample-1.sentences_hi.txt` and try to spot a sentence that looks
  out of order. This is what a low Kendall tau looks like in practice.
- Run the pipeline with `--strategy clusters` and compare the result with
  `--strategy borda`.
- Change one sentence in a copy of a raw file and watch how the output changes.
- Open `reports/evaluation.json` and find the per-class punctuation F1 scores.
  Explain why `question` and `exclamation` are difficult.

## 8. Troubleshooting

| Problem | What to do |
| --- | --- |
| `FileNotFoundError` for a model | Train the adapter first, or use the full Colab notebook. |
| `IndicTransToolkit` import error | Run `bash scripts/colab_setup.sh`. |
| `401` for IndicTrans2 | Accept the licence and log in to Hugging Face, or pass a local model path. |
| CUDA/MPS out of memory | Close other notebooks, or run one stage at a time. |
| Import error for `src` | Run cell 0 again from the project root. |

## Glossary

| Term | Meaning |
| --- | --- |
| **Adapter** | Saved LoRA weights on top of a frozen base model. |
| **BLEU / chrF** | Translation quality scores; higher is better. |
| **Borda** | A decoder that averages "before the others" scores and sorts. |
| **Inference** | Using a trained model to make predictions. |
| **Kendall's tau** | A score from -1 to 1 measuring order similarity. |
| **Label** | The answer for one word. |
| **LoRA** | Small trainable weights added to a frozen model. |
| **mBERT** | The multilingual BERT model used for stages 1 and 2. |
| **Subword** | A piece of a word produced by the tokenizer. |
| **Token** | The smallest text unit a model reads. |